# Generador de Gráficos 2.5 (Estandarizado Villa Gaucho)

Este cuaderno es una versión estandarizada de `generador_graficos 2_5.ipynb`, aplicando las especificaciones de diseño y formato del Informe de Villa Gaucho:
- **Tipografía y tamaño de letra 14**: configurado globalmente para títulos, etiquetas de ejes, valores y señalizaciones.
- **Gráficos de Torta (Cualitativas)**:
  - Sin líneas blancas divisorias (`edgecolor='none'`).
  - Frecuencia absoluta adentro de cada cuña (tamaño 14).
  - Línea horizontal divisoria que arranca exactamente en el inicio del texto.
  - Nombre de la categoría arriba de la línea y porcentaje abajo (`X,X%`).
  - Para preguntas binarias (ej. Tiene motivo Sí / No): **Positivo = Azul (`#4285F4`)**, **Negativo = Naranja (`#FB8C00`)**.
  - Para razones/motivos múltiples: orden de mayor a menor porcentaje con la paleta de colores: 1. Azul (`#4285F4`), 2. Naranja (`#FB8C00`), 3. Verde (`#34A853`), 4. Amarillo (`#FBBC05`), 5. Violeta (`#8E24AA`), 6. Beige (`#D7CCC8`).
  - Distribución balanceada verticalmente en el recuadro para evitar superposiciones.


In [ ]:
# ==============================================================================
# 1. IMPORTACIÓN DE LIBRERÍAS Y CONFIGURACIÓN GLOBAL
# ==============================================================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Configuración global de tamaño de letra 14
plt.rcParams['font.size'] = 14
plt.rcParams['axes.titlesize'] = 14
plt.rcParams['axes.labelsize'] = 14
plt.rcParams['xtick.labelsize'] = 14
plt.rcParams['ytick.labelsize'] = 14
plt.rcParams['legend.fontsize'] = 14
plt.rcParams['figure.titlesize'] = 14

# Paleta para tortas (de mayor a menor porcentaje)
PALETA_TORTA = [
    '#4285F4',  # 1. Azul
    '#FB8C00',  # 2. Naranja
    '#34A853',  # 3. Verde
    '#FBBC05',  # 4. Amarillo
    '#8E24AA',  # 5. Violeta
    '#D7CCC8'   # 6. Beige
]

# Colores para preguntas binarias (Positivo / Negativo)
COLOR_POSITIVO = '#4285F4'  # Azul
COLOR_NEGATIVO = '#FB8C00'  # Naranja


In [ ]:
# ==============================================================================
# 2. FUNCIÓN DE TORTA ESTANDARIZADA
# ==============================================================================
def graficar_torta(categorias, valores, titulo="", es_binario=False, orden_positivo_primero=True, figsize=(11, 7), distancia_etiquetas_y=0.45):
    """
    Genera un gráfico de torta con el formato del Informe Villa Gaucho:
    - Sin bordes blancos entre porciones.
    - Frecuencia absoluta adentro de cada cuña (tamaño 14).
    - Línea conector horizontal que arranca justo en el texto.
    - Nombre de la categoría arriba de la línea, porcentaje abajo.
    - Si es binario (Sí/No): Positivo = Azul, Negativo = Naranja.
    - Si es multicategoría: orden de mayor a menor con paleta institucional.
    - Distribución balanceada verticalmente en el recuadro.
    """
    df_temp = pd.DataFrame({'categoria': [str(c) for c in categorias], 'fi': list(valores)})

    if es_binario:
        def es_pos(val):
            val_clean = str(val).strip().lower()
            return val_clean in ['si', 'sí', 'positivo', 'true', 'afirma', 'posee', 'tiene', 'concurre']

        df_temp['es_pos'] = df_temp['categoria'].apply(es_pos)
        if orden_positivo_primero:
            df_temp = df_temp.sort_values(by='es_pos', ascending=False).reset_index(drop=True)
        colores = [COLOR_POSITIVO if p else COLOR_NEGATIVO for p in df_temp['es_pos']]
    else:
        df_temp = df_temp.sort_values(by='fi', ascending=False).reset_index(drop=True)
        colores = [PALETA_TORTA[i % len(PALETA_TORTA)] for i in range(len(df_temp))]

    cats = df_temp['categoria'].tolist()
    vals = df_temp['fi'].values
    total = sum(vals)
    pcts = (vals / total) * 100 if total > 0 else np.zeros_like(vals)

    fig, ax = plt.subplots(figsize=figsize, subplot_kw=dict(aspect="equal"))

    wedges, texts, autotexts = ax.pie(
        vals,
        autopct='%1.0f',
        pctdistance=0.68,
        colors=colores,
        startangle=90,
        counterclock=False,
        wedgeprops=dict(width=1, edgecolor='none')
    )

    for at, val in zip(autotexts, vals):
        at.set_text(str(int(val)))
        at.set_color('#202124')
        at.set_fontsize(14)
        at.set_fontweight('medium')

    wedge_items = []
    for i, p in enumerate(wedges):
        ang = (p.theta2 - p.theta1) / 2.0 + p.theta1
        rad = np.deg2rad(ang)
        x = np.cos(rad)
        y = np.sin(rad)
        lado = 1 if x >= 0 else -1
        wedge_items.append({
            'index': i,
            'ang': ang,
            'rad': rad,
            'x': x,
            'y': y,
            'lado': lado,
            'y_afuera': 1.15 * y
        })

    ax.set_xlim(-2.4, 2.4)
    ax.set_ylim(-1.6, 1.6)

    # Distribución balanceada vertical para evitar superposiciones
    for lado in [1, -1]:
        group = [item for item in wedge_items if item['lado'] == lado]
        if not group:
            continue
        group.sort(key=lambda item: item['y'])
        k = len(group)
        if k == 1:
            group[0]['y_afuera'] = group[0]['y'] * 1.15
        else:
            span = (k - 1) * distancia_etiquetas_y
            avg_y = np.mean([item['y'] for item in group])
            start_y = avg_y - (span / 2.0)
            start_y = max(-1.30, min(1.30 - span, start_y))
            for idx_g, item in enumerate(group):
                item['y_afuera'] = start_y + idx_g * distancia_etiquetas_y

    plt.draw()
    inv = ax.transData.inverted()

    for item in wedge_items:
        i = item['index']
        rad = item['rad']
        x_punto = 0.90 * np.cos(rad)
        y_punto = 0.90 * np.sin(rad)

        lado = item['lado']
        y_afuera = item['y_afuera']
        ha = 'left' if lado == 1 else 'right'
        pct_str = f"{pcts[i]:.1f}%".replace('.', ',')

        x_margen = 2.05 * lado

        t_cat = ax.text(
            x_margen, y_afuera + 0.03,
            cats[i],
            horizontalalignment=ha,
            verticalalignment='bottom',
            fontsize=14,
            color='#202124',
            fontweight='medium'
        )
        t_pct = ax.text(
            x_margen, y_afuera - 0.03,
            pct_str,
            horizontalalignment=ha,
            verticalalignment='top',
            fontsize=14,
            color='#757575',
            fontweight='normal'
        )

        plt.draw()
        bbox_cat = inv.transform(t_cat.get_window_extent(fig.canvas.get_renderer()))
        bbox_pct = inv.transform(t_pct.get_window_extent(fig.canvas.get_renderer()))
        w_cat = abs(bbox_cat[1][0] - bbox_cat[0][0])
        w_pct = abs(bbox_pct[1][0] - bbox_pct[0][0])
        w_max = max(w_cat, w_pct, 0.40)

        # La línea arranca exactamente en el inicio del texto
        if lado == 1:
            x_codo = x_margen
            x_fin = x_margen + w_max
            ax.plot([x_punto, x_codo, x_fin], [y_punto, y_afuera, y_afuera], color='#9e9e9e', lw=1.2)
        else:
            x_codo = x_margen
            x_inicio = x_margen - w_max
            ax.plot([x_punto, x_codo, x_inicio], [y_punto, y_afuera, y_afuera], color='#9e9e9e', lw=1.2)

        ax.plot(x_punto, y_punto, marker='o', markersize=4, color='#9e9e9e')

    if titulo:
        ax.set_title(titulo, fontsize=14, pad=25, fontweight='bold', color='#202124')

    plt.subplots_adjust(left=0.08, right=0.92, top=0.88, bottom=0.10)
    plt.show()


## 3. Carga y Procesamiento de Datos


In [ ]:
# Cargar el archivo CSV y limpiar espacios en los nombres de columnas
df = pd.read_csv('datos.csv')
df.columns = df.columns.str.strip()

col_motivo = '2.5 En caso de que algún integrante no realice todas las comidas, ¿a qué se debe? Indicar la respuesta de manera que pueda identificarse según el integrante. Ejemplo: Int. 1 (motivo) - Int. 2 (motivo) Definido'

# Dataset base y variable indicadora ('Si' / 'No')
df_2_5_combinado = df[[col_motivo]].copy()
df_2_5_combinado = df_2_5_combinado.rename(columns={col_motivo: 'Motivo'})
df_2_5_combinado['Tiene_motivo'] = np.where(df_2_5_combinado['Motivo'].notna(), 'Si', 'No')

# Frecuencia de la variable indicadora binaria
df_2_5_indicadorfi = df_2_5_combinado['Tiene_motivo'].value_counts().to_frame(name='fi')

# Dataset solo con motivos reportados
df_2_5_razones = df_2_5_combinado[df_2_5_combinado['Tiene_motivo'] == 'Si'][['Motivo']].reset_index(drop=True)

# Estandarización de las razones
razones_map = {
    'escuela-trabajo': 'Escuela / Trabajo',
    'salud-dieta': 'Salud y Dieta',
    'preferencia': 'Preferencia Personal',
    'motivos economicos': 'Motivos Económicos'
}

conteos = {
    nombre_limpio: df_2_5_razones['Motivo'].str.contains(clave, case=False, na=False).sum()
    for clave, nombre_limpio in razones_map.items()
}

df_2_5_estandarizado = pd.DataFrame(list(conteos.items()), columns=['Razon', 'fi'])


## 4. Pregunta 2.5: Hogares con motivos para no realizar todas las comidas


In [ ]:
# Gráfico de torta: Pregunta binaria Sí / No (Positivo = Azul, Negativo = Naranja)
graficar_torta(
    categorias=df_2_5_indicadorfi.index,
    valores=df_2_5_indicadorfi['fi'],
    titulo="¿Algún integrante no realiza todas las comidas en el hogar?",
    es_binario=True  # Positivo = Azul, Negativo = Naranja
)


## 5. Pregunta 2.5: Motivos reportados


In [ ]:
# Gráfico de torta: Distribución de motivos (Variable Cualitativa Multicategoría)
graficar_torta(
    categorias=df_2_5_estandarizado['Razon'],
    valores=df_2_5_estandarizado['fi'],
    titulo="Motivos por los cuales no se realizan todas las comidas",
    distancia_etiquetas_y=0.45
)
